# Modelado — comparación de modelos (forecast t+1)

Comparación de **regresión lineal**, **Random Forest** y **XGBoost** para predecir
`prod_pet` del mes siguiente (`y_next`).

**Protocolo (ADR-028):**
- Se **entrena en `train`** y se **compara en `val`**; `test` queda intacto.
- Métricas: **RMSE** (m³, penaliza errores grandes) y **R²** (varianza explicada).
- Todo el preprocesamiento (one-hot, imputación, normalización) se **ajusta solo
  en train** y se aplica a val → sin leakage val→train. Las features ya son
  leak-safe en el tiempo (ver notebook 03).

La lógica reutilizable vive en `ml/modeling.py`; acá solo se orquesta.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ml import modeling

pd.set_option("display.width", 120)

## 1. Dataset y matriz de features

`build_feature_matrix` arma el dataset (notebook 03) y aplica el one-hot de las
categóricas **ajustado en train**. Las features = todas las columnas menos las
claves (`idpozo`, `periodo`, `periodo_objetivo`, `split`) y el target `y_next`.

In [2]:
ds_enc, feature_cols = modeling.build_feature_matrix()

print(f"Filas: {len(ds_enc):,}  |  Features: {len(feature_cols)}")
print(ds_enc["split"].value_counts().reindex(["train", "val", "test"]).to_string())

Filas: 319,554  |  Features: 29
split
train    223615
val       47883
test      48056


## 2. Split train/val + preprocesamiento (anti-leakage)

- **Imputación** de NaN con la **mediana de train** (Random Forest y regresión
  lineal no toman NaN). Los NaN vienen de las features de historia (primeros
  meses de cada pozo).
- **Normalización** (media 0, desvío 1) **solo para la regresión lineal**
  (sensible a la escala); los árboles no la necesitan.

Ambos estadísticos (mediana, media/desvío) se ajustan **solo en train**.

In [3]:
X_train, y_train, X_val, y_val = modeling.split_train_val(ds_enc, feature_cols)
print("train:", X_train.shape, " | val:", X_val.shape)

train: (223615, 29)  | val: (47883, 29)


## 3. Entrenamiento y comparación en val

`train_eval_models` entrena los 3 modelos en train (imputado; + normalizado para
el lineal) y los evalúa en val. Agregamos la **persistencia** (ŷ(t+1) = prod_pet(t))
como referencia: es el baseline a batir (ADR-029).

In [4]:
tabla, modelos = modeling.train_eval_models(X_train, y_train, X_val, y_val)

pers = modeling.persistence_val(ds_enc)
comparacion = pd.concat(
    [tabla, pd.DataFrame([{"modelo": "persistencia (baseline)", **pers}])],
    ignore_index=True,
).sort_values("val_rmse").reset_index(drop=True)
comparacion.round(3)

,modelo,val_rmse,val_r2
0,xgboost,238.538,0.893
1,reg_lineal,242.818,0.889
2,persistencia (baseline),250.589,0.881
3,random_forest,251.870,0.880


## 4. Tuning de hiperparámetros con CV temporal

¿Vale la pena un grid search? **Sí**, pero **no** con el `KFold` por defecto: baraja
las filas al azar, así que folds podrían entrenar con meses *posteriores* a los que
validan → **leakage futuro→pasado**. Por ser serie temporal usamos:

- **CV temporal (expanding window) por mes** (`modeling.time_series_folds`): en cada
  fold el bloque de validación es **posterior** a todo el train del fold.
- **`Pipeline` (imputer + scaler) dentro del search**: `GridSearchCV` reajusta el
  preprocesamiento **solo con el train de cada fold** → sin leakage de estadísticos.
- La búsqueda se hace **dentro de train**; **val sigue intacto** como comparación final.

Modelos: **Ridge** (tuneando `alpha` = lambda L2), **Random Forest** y **XGBoost**.

Cada modelo muestra una **barra de progreso** (`tqdm`) que avanza por cada
combinación de hiperparámetros e indica el mejor RMSE de CV hasta el momento.

> ⚠️ Esta celda entrena muchos modelos (sobre todo el Random Forest): puede tardar
> varios minutos. La barra te deja ver cómo evoluciona.

In [5]:
tabla_cv, searches = modeling.tune_all(ds_enc, feature_cols, n_splits=4)
tabla_cv

ridge (20 configs):   0%|          | 0/20 [00:00<?, ?it/s]

random_forest (20 configs):   0%|          | 0/20 [00:00<?, ?it/s]

xgboost (20 configs):   0%|          | 0/20 [00:00<?, ?it/s]

,modelo,cv_rmse,best_params
0,random_forest,213.215365,"{'model__n_estimators': 400, 'model__min_sampl..."
1,ridge,213.589544,{'model__alpha': 1128.8378916846884}
2,xgboost,227.958929,"{'model__subsample': 0.7, 'model__n_estimators..."


Mejores hiperparámetros por modelo (los de menor RMSE en la CV temporal):

In [6]:
for nombre, search in searches.items():
    print(f"{nombre}: cv_rmse={-search.best_score_:.3f}")
    print("  ", search.best_params_)

ridge: cv_rmse=213.590
   {'model__alpha': 1128.8378916846884}
random_forest: cv_rmse=213.215
   {'model__n_estimators': 400, 'model__min_samples_leaf': 2, 'model__max_features': 0.5, 'model__max_depth': 16}
xgboost: cv_rmse=227.959
   {'model__subsample': 0.7, 'model__n_estimators': 400, 'model__min_child_weight': 10, 'model__max_depth': 12, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1.0}


### 4.1 Evaluación de los modelos tuneados en val

Cada `best_estimator_` (reentrenado en **todo** train con sus mejores
hiperparámetros) se evalúa sobre **val**, que no se tocó durante el tuning. Comparamos
contra la persistencia y, en la sección 3, contra los modelos sin tunear.

In [7]:
comparacion_tuned = modeling.eval_searches_on_val(searches, ds_enc, feature_cols)
comparacion_tuned.round(3)

,modelo,val_rmse,val_r2
0,random_forest,229.864,0.900
1,ridge,241.805,0.890
2,xgboost,244.293,0.887
3,persistencia (baseline),250.589,0.881


### 4.2 Métricas en train y val con la mejor config

Toma la **mejor configuración** hallada por modelo (el `best_estimator_`, ya
reentrenado en todo train) y mide RMSE/R² en **train** y en **val**. Ver las dos
juntas muestra el **gap train→val**: si train es mucho mejor que val, el modelo
está sobreajustando.

In [8]:
metricas_train_val = modeling.eval_searches_train_val(searches, ds_enc, feature_cols)
metricas_train_val.round(3)

,modelo,train_rmse,train_r2,val_rmse,val_r2
0,random_forest,132.860,0.972,229.864,0.900
1,ridge,278.120,0.879,241.805,0.890
2,xgboost,156.875,0.961,244.293,0.887
3,persistencia (baseline),309.994,0.849,250.589,0.881
